In [18]:
import pyedflib
import numpy as np
import os 
import re
import pandas as pd

In [14]:
def find_file_pairs(root_dir, verbose=False):
    """
    More flexible pairing:
    - case-insensitive
    - accepts filenames containing 'psg' (not only '-PSG.edf')
    - matches hypnogram files by subject id or by existence in the same directory
    """
    file_pairs = []
    id_re = re.compile(r'(SC\d{3,4}E\d)', re.IGNORECASE)

    for dirpath, _, filenames in os.walk(root_dir):
        psgs = [fn for fn in filenames if fn.lower().endswith('.edf') and 'psg' in fn.lower()]
        hyps = [fn for fn in filenames if 'hypnogram' in fn.lower()]

        if verbose and (psgs or hyps):
            print(f"[{dirpath}] PSGs={len(psgs)} Hyps={len(hyps)}")

        for psg in psgs:
            psg_path = os.path.join(dirpath, psg)
            m = id_re.search(psg)
            candidate_hyps = []

            if m:
                sid = m.group(1)
                candidate_hyps = [h for h in hyps if sid.lower() in h.lower()]

            if not candidate_hyps:
                alt_name = re.sub(r'(?i)psg', 'Hypnogram', psg)
                if alt_name in filenames:
                    candidate_hyps = [alt_name]
                else:
                    candidate_hyps = hyps[:]  # any hyp in same dir

            for h in candidate_hyps:
                hyp_path = os.path.join(dirpath, h)
                if os.path.exists(hyp_path):
                    file_pairs.append((psg_path, hyp_path))

    seen = set()
    deduped = []
    for a, b in file_pairs:
        key = (a.lower(), b.lower())
        if key not in seen:
            seen.add(key)
            deduped.append((a, b))

    return deduped

In [15]:
data_dir = r"E:\Github\SleepBud-Machine-Learning\sleep-edf-database-expanded-1.0.0\sleep-edf-database-expanded-1.0.0\sleep-cassette"
all_files = find_file_pairs(data_dir, verbose=True)
print(f"Found {len(all_files)} pairs of files.")

[E:\Github\SleepBud-Machine-Learning\sleep-edf-database-expanded-1.0.0\sleep-edf-database-expanded-1.0.0\sleep-cassette] PSGs=153 Hyps=153
Found 23409 pairs of files.


In [12]:
%pip list

Package                 Version
----------------------- -----------
asttokens               3.0.0
colorama                0.4.6
comm                    0.2.3
contourpy               1.3.3
cycler                  0.12.1
debugpy                 1.8.17
decorator               5.2.1
executing               2.2.1
fonttools               4.60.1
ipykernel               7.0.1
ipython                 9.6.0
ipython_pygments_lexers 1.1.1
jedi                    0.19.2
jupyter_client          8.6.3
jupyter_core            5.9.1
kiwisolver              1.4.9
matplotlib              3.10.7
matplotlib-inline       0.1.7
nest-asyncio            1.6.0
numpy                   2.3.4
packaging               25.0
pandas                  2.3.3
parso                   0.8.5
pillow                  12.0.0
pip                     25.2
platformdirs            4.5.0
prompt_toolkit          3.0.52
psutil                  7.1.0
pure_eval               0.2.3
pyEDFlib                0.1.42
Pygments                2.

In [20]:
def read_edf_data(psg_file, hypnogram_file):
    # Read the PSG signal file
    psg = pyedflib.EdfReader(psg_file)
    
    # Find the EMG signal channel (proxy for movement)
    emg_channel_index = -1
    for i in range(psg.signals_in_file):
        if "EMG" in psg.getSignalHeader(i)['label']:
            emg_channel_index = i
            break
            
    if emg_channel_index == -1:
        return None, None, None

    emg_signal = psg.readSignal(emg_channel_index)
    # Corrected the key here
    sampling_rate = psg.getSignalHeader(emg_channel_index)['sample_frequency']
    psg.close()

    # Read the hypnogram label file
    hypnogram = pyedflib.EdfReader(hypnogram_file)
    annotations = hypnogram.readAnnotations()
    hypnogram.close()
    
    return emg_signal, annotations, sampling_rate

In [22]:
def process_session(psg_file, hypnogram_file):
    emg_signal, annotations, sampling_rate = read_edf_data(psg_file, hypnogram_file)

    if emg_signal is None:
        return []

    # The labels in this dataset are for 30-second epochs
    epoch_duration = 30 
    samples_per_epoch = int(sampling_rate * epoch_duration)
    
    processed_data = []
    
    # The annotations contain (onset, duration, label)
    # We only care about the label for each 30-second window
    labels = annotations[2] 
    
    for i, label in enumerate(labels):
        # Get the 30-second chunk of the signal corresponding to the label
        start_sample = i * samples_per_epoch
        end_sample = start_sample + samples_per_epoch
        
        if end_sample > len(emg_signal):
            break
            
        signal_window = emg_signal[start_sample:end_sample]
        
        # Calculate the feature for this window
        emg_variance = np.var(signal_window)
        
        # Store the feature and its label
        processed_data.append({
            'emg_variance': emg_variance,
            'sleep_stage': label
        })
        
    return processed_data

In [23]:
all_features = []
for psg_file, hypnogram_file in all_files:
    print(f"Processing {os.path.basename(psg_file)}...")
    session_features = process_session(psg_file, hypnogram_file)
    all_features.extend(session_features)

print(f"\nExtracted features for {len(all_features)} total epochs.")

# Convert to a pandas DataFrame
df = pd.DataFrame(all_features)

# Clean up the labels
# Define a mapping from the file's labels to a simpler set
label_map = {
    'Sleep stage W': 0, # Wake
    'Sleep stage 1': 1, # Light Sleep (N1)
    'Sleep stage 2': 2, # Light Sleep (N2)
    'Sleep stage 3': 3, # Deep Sleep (N3)
    'Sleep stage 4': 3, # Also Deep Sleep (merge with N3)
    'Sleep stage R': 4, # REM
}

# Apply the mapping and remove unknown stages
df['sleep_stage_encoded'] = df['sleep_stage'].map(label_map)
df = df.dropna() # Removes any stages not in our map (like 'Movement time' or '?')
df['sleep_stage_encoded'] = df['sleep_stage_encoded'].astype(int)


# Save the final, clean dataset
df.to_csv('sleep_model_training_data.csv', index=False)

print("\nFinal dataset saved to 'sleep_model_training_data.csv'")
print("Head of the final dataset:")
print(df[['emg_variance', 'sleep_stage_encoded']].head())

Processing SC4001E0-PSG.edf...
Processing SC4001E0-PSG.edf...
Processing SC4001E0-PSG.edf...
Processing SC4001E0-PSG.edf...
Processing SC4001E0-PSG.edf...
Processing SC4001E0-PSG.edf...
Processing SC4001E0-PSG.edf...
Processing SC4001E0-PSG.edf...
Processing SC4001E0-PSG.edf...
Processing SC4001E0-PSG.edf...
Processing SC4001E0-PSG.edf...
Processing SC4001E0-PSG.edf...
Processing SC4001E0-PSG.edf...
Processing SC4001E0-PSG.edf...
Processing SC4001E0-PSG.edf...
Processing SC4001E0-PSG.edf...
Processing SC4001E0-PSG.edf...
Processing SC4001E0-PSG.edf...
Processing SC4001E0-PSG.edf...
Processing SC4001E0-PSG.edf...
Processing SC4001E0-PSG.edf...
Processing SC4001E0-PSG.edf...
Processing SC4001E0-PSG.edf...
Processing SC4001E0-PSG.edf...
Processing SC4001E0-PSG.edf...
Processing SC4001E0-PSG.edf...
Processing SC4001E0-PSG.edf...
Processing SC4001E0-PSG.edf...
Processing SC4001E0-PSG.edf...
Processing SC4001E0-PSG.edf...
Processing SC4001E0-PSG.edf...
Processing SC4001E0-PSG.edf...
Processi